# Notebook 03 — DPO Trials

**Prerequisite:** Notebook 02 must have finished and written `results/sft_winner.json`.

**Inputs:** `results/sft_winner.json`, `models/sft_<winner>/`, `configs/dpo_trials.json`,
`data/test_prompts.json`, `data/gold_answers.json`  
**Outputs:** `results/dpo_<name>.json` × 5, `results/dpo_winner.json`, `models/dpo_<name>/` × 5  
**Runtime:** ~2–5 hours on Kaggle T4

---
⚠ **Crash recovery:** Already-finished trials are skipped automatically on re-run.

⚠ **Parallel mode:** Set `TRIALS_TO_RUN = [0,1,2]` in Session A and `[3,4]` in Session B.
Set `SELECT_WINNER = False` in both. Re-run once all finish to pick winner.

## Cell 1 — Configuration (edit here for parallel sessions)

In [ ]:
# ─── EDIT THESE TWO LINES FOR PARALLEL SESSIONS ───────────────────────────
TRIALS_TO_RUN = [0, 1, 2, 3, 4]   # Change to [0,1,2] or [3,4] for parallel split
SELECT_WINNER = True               # Set False when running a subset
# ───────────────────────────────────────────────────────────────────────────

print(f"Will run trial indices: {TRIALS_TO_RUN}")
print(f"Select winner at end:   {SELECT_WINNER}")

## Cell 2 — Install dependencies

In [ ]:
import subprocess, sys

PACKAGES = [
    "transformers>=4.45.0", "peft>=0.13.0", "trl>=0.11.0",
    "bitsandbytes>=0.44.0", "accelerate>=1.0.0", "datasets>=3.0.0",
    "sacrebleu", "bert-score",
]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers", "huggingface_hub"], check=False)
print("Done.")

## Cell 3 — Paths & HF login

In [ ]:
import os, sys, json, time
from pathlib import Path
import torch

KAGGLE = Path("/kaggle").exists()
if KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working/daa-helper")
    if not PROJECT_ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone",
                        "https://github.com/hashirilyas1803/daa-helper.git",
                        str(PROJECT_ROOT)], check=False)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

from huggingface_hub import login
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste HF token: ").strip()
login(token=HF_TOKEN)

HF_USERNAME = "hashirilyas18"
BASE_MODEL  = "TinyLlama/TinyLlama_v1.1"

from utils.io_helpers import load_json, save_trial_result, list_existing_results
from utils.evaluation  import run_inference_on_prompts, evaluate_responses, free_memory
print("Setup complete.")

## Cell 4 — Load winner info, configs, prompts, gold answers

In [ ]:
winner_info  = load_json("sft_winner.json",   base_dir="results")
prompts_data = load_json("test_prompts.json",  base_dir="data")
gold_data    = load_json("gold_answers.json",  base_dir="data")
dpo_config   = load_json("dpo_trials.json",    base_dir="configs")

SFT_WINNER_NAME = winner_info["winning_trial"]
SFT_WINNER_DIR  = str(PROJECT_ROOT / "models" / f"sft_{SFT_WINNER_NAME}")
assert Path(SFT_WINNER_DIR).exists(), \
    f"SFT adapter not found: {SFT_WINNER_DIR}. Run Notebook 02 first."

prompts      = [p["prompt"]      for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]
all_trials   = dpo_config["trials"]

# Apply subset filter from Cell 1
trials = [all_trials[i] for i in TRIALS_TO_RUN if i < len(all_trials)]

print(f"SFT winner: {SFT_WINNER_NAME}")
print(f"Running {len(trials)} of {len(all_trials)} DPO trials")

## Cell 5 — Load DPO preference dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading trl-lib/ultrafeedback_binarized...")
raw = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
raw = raw.shuffle(seed=42).select(range(min(2000, len(raw))))
print(f"Loaded {len(raw)} preference pairs | Columns: {raw.column_names}")

split        = int(0.9 * len(raw))
dpo_train_ds = raw.select(range(split))
dpo_eval_ds  = raw.select(range(split, len(raw)))
print(f"Train: {len(dpo_train_ds)}  Eval: {len(dpo_eval_ds)}")

## Cell 6 — Define training & evaluation helpers

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl  import DPOTrainer, DPOConfig


def load_sft_winner():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    base  = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map="auto",
        trust_remote_code=True, torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, SFT_WINNER_DIR, is_trainable=True)
    return model


def run_dpo_trial(trial, train_ds, eval_ds, output_dir):
    print(f"\n{'='*60}\nDPO TRIAL: {trial['name']}\n{'='*60}")
    print(f"  beta={trial['beta']}  lr={trial['learning_rate']}")
    model = load_sft_winner()

    args = DPOConfig(
        output_dir=output_dir,
        num_train_epochs=trial["num_train_epochs"],
        per_device_train_batch_size=trial["per_device_train_batch_size"],
        gradient_accumulation_steps=trial["gradient_accumulation_steps"],
        per_device_eval_batch_size=trial["per_device_train_batch_size"],
        learning_rate=trial["learning_rate"],
        beta=trial["beta"],
        max_length=trial["max_length"],
        max_prompt_length=trial["max_prompt_length"],
        logging_steps=5, eval_strategy="steps", eval_steps=20,
        save_strategy="no", warmup_ratio=0.05,
        lr_scheduler_type="cosine", bf16=True,
        optim="paged_adamw_8bit", report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
    )
    trainer = DPOTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        tokenizer=tokenizer,
    )
    start  = time.time()
    result = trainer.train()
    elapsed = time.time() - start

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    eval_metrics = trainer.evaluate()

    train_metrics = {
        "train_loss":            result.training_loss,
        "eval_loss":             eval_metrics.get("eval_loss"),
        "rewards_chosen":        eval_metrics.get("eval_rewards/chosen"),
        "rewards_rejected":      eval_metrics.get("eval_rewards/rejected"),
        "rewards_margin":        eval_metrics.get("eval_rewards/margins"),
        "train_runtime_seconds": elapsed,
    }
    del trainer, model
    free_memory()
    return train_metrics


def evaluate_adapter(adapter_dir):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    base  = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map="auto",
        trust_remote_code=True, torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    responses    = run_inference_on_prompts(model, tokenizer, prompts, max_new_tokens=512)
    eval_results = evaluate_responses(responses, gold_answers)
    samples      = [{"prompt": p, "response": r, "gold": g}
                    for p, r, g in zip(prompts, responses, gold_answers)]
    del model, base
    free_memory()
    return eval_results, samples


print("Helpers defined.")

## Cell 7 — Run trials

Already-finished trials are skipped automatically.

In [ ]:
existing = set(list_existing_results(stage="dpo"))
print(f"Existing DPO results: {existing or '(none)'}")

for trial in trials:
    fname = f"dpo_{trial['name']}.json"
    if fname in existing:
        print(f"\n✓ Skipping {trial['name']} (already done)")
        continue

    out_dir = str(PROJECT_ROOT / "models" / f"dpo_{trial['name']}")
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    try:
        train_metrics = run_dpo_trial(trial, dpo_train_ds, dpo_eval_ds, out_dir)
        eval_results, samples = evaluate_adapter(out_dir)
        save_trial_result(
            trial_name=trial["name"], stage="dpo", config=trial,
            eval_results=eval_results, train_metrics=train_metrics,
            sample_responses=samples,
        )
        agg = eval_results["aggregate"]
        print(f"✓ {trial['name']}")
        print(f"  BLEU={agg['mean_bleu']:.2f}  "
              f"BERTScore={agg['mean_bertscore_f1']:.4f}  "
              f"EvalLoss={train_metrics['eval_loss']:.4f}")

    except Exception as e:
        import traceback
        print(f"\n✗ ERROR in {trial['name']}: {e}")
        traceback.print_exc()
        free_memory()

print("\nAll assigned DPO trials done.")

## Cell 8 — Select winner

Skip this cell if `SELECT_WINNER = False`.

In [ ]:
if not SELECT_WINNER:
    print("SELECT_WINNER=False — skipping. Re-run once all sessions finish.")
else:
    results = []
    for trial in all_trials:
        try:
            results.append(load_json(f"dpo_{trial['name']}.json", base_dir="results"))
        except FileNotFoundError:
            print(f"  ⚠ Missing result for {trial['name']}")

    results.sort(key=lambda r: (
        -r["evaluation"]["aggregate"]["combined_score"],
         r["training_metrics"].get("eval_loss") or float("inf"),
    ))

    print(f"{'Trial':<30} {'BLEU':>7} {'BERTScore':>10} {'Combined':>10} {'EvalLoss':>10}")
    for r in results:
        a  = r["evaluation"]["aggregate"]
        el = r["training_metrics"].get("eval_loss", float("nan"))
        print(f"{r['trial_name']:<30} {a['mean_bleu']:>7.2f} "
              f"{a['mean_bertscore_f1']:>10.4f} {a['combined_score']:>10.4f} {el:>10.4f}")

    winner = results[0]
    print(f"\n🏆 DPO WINNER: {winner['trial_name']}")

    with open(PROJECT_ROOT / "results" / "dpo_winner.json", "w") as f:
        json.dump({
            "winning_trial":   winner["trial_name"],
            "winning_config":  winner["config"],
            "winning_metrics": winner["evaluation"]["aggregate"],
        }, f, indent=2)
    print("Saved → results/dpo_winner.json")

## Cell 9 — Push results to GitHub & adapter to HF Hub

In [ ]:
import subprocess

if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    GITHUB_TOKEN = sec.get_secret("GITHUB_TOKEN")
    GITHUB_USER  = sec.get_secret("GITHUB_USER")

    # Push results JSON
    cmds = [
        ["git", "config", "--global", "user.email", "you@example.com"],
        ["git", "config", "--global", "user.name", GITHUB_USER],
        ["git", "-C", str(PROJECT_ROOT), "add", "results/"],
        ["git", "-C", str(PROJECT_ROOT), "commit", "-m", "dpo results"],
        ["git", "-C", str(PROJECT_ROOT), "push",
         f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/daa-helper.git"],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True)
        out = (r.stdout or r.stderr).strip()
        if out: print(out)

    # Push winning DPO adapter to HF Hub
    if SELECT_WINNER and 'winner' in dir():
        from huggingface_hub import HfApi
        api     = HfApi()
        repo_id = f"{HF_USERNAME}/daa-helper-tinyllama-dpo-winner"
        api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(
            folder_path=str(PROJECT_ROOT / "models" / f"dpo_{winner['trial_name']}"),
            repo_id=repo_id, repo_type="model",
            commit_message=f"DPO winner: {winner['trial_name']}",
        )
        print(f"Pushed adapter → {repo_id}")
else:
    print("Not on Kaggle — skipping push.")

print("\n✓ Notebook 03 complete. Run Notebook 04 next.")